## Подготовка данных

Используем датасет **Titanic** — задачу бинарной классификации со смешанными признаками.

**Признаки:**
- pclass: класс пассажира (числовой)
- sex: пол (категориальный)
- age: возраст (числовой)
- sibsp: число родственников на борту (числовой)
- fare: стоимость билета (числовой)
- embarked: порт посадки (категориальный)

**Целевая переменная:** survived (0 = no, 1 = yes)

In [16]:
from model import DecisionTree
from collections import Counter
import csv
import urllib.request

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
response = urllib.request.urlopen(url, timeout=10)
raw_data = response.read().decode("utf-8")
rows = list(csv.DictReader(raw_data.splitlines()))
print(f"Загружено {len(rows)} записей Titanic")

def to_float(value):
    if value is None or value == "":
        return None
    return float(value)

X = []
y = []
for row in rows:
    X.append([
        int(row["Pclass"]),
        row["Sex"],
        to_float(row["Age"]),
        int(row["SibSp"]),
        to_float(row["Fare"]),
        row["Embarked"] if row["Embarked"] else None,
    ])
    y.append(int(row["Survived"]))

feature_types = {
    0: 'numeric',
    1: 'categorical',
    2: 'numeric',
    3: 'numeric',
    4: 'numeric',
    5: 'categorical',
}

feature_names = ['Pclass', 'Sex', 'Age', 'SibSp', 'Fare', 'Embarked']
class_names = {0: 'Did not survive', 1: 'Survived'}

print(f"\nРазмер датасета: {len(X)} образцов")
print(f"Количество признаков: {len(X[0])}")
print(f"Категориальных: {sum(1 for t in feature_types.values() if t == 'categorical')}")
print(f"Числовых: {sum(1 for t in feature_types.values() if t == 'numeric')}")
print(f"Классы: {set(y)}")
print("Распределение классов:")
for cls, name in class_names.items():
    count = sum(1 for label in y if label == cls)
    print(f"  {cls} ({name}): {count} ({100*count/len(y):.1f}%)")

Загружено 891 записей Titanic

Размер датасета: 891 образцов
Количество признаков: 6
Категориальных: 2
Числовых: 4
Классы: {0, 1}
Распределение классов:
  0 (Did not survive): 549 (61.6%)
  1 (Survived): 342 (38.4%)


In [2]:
def train_test_split(X, y, test_size=0.2, random_state=None):
    """Split data into train and test sets."""
    n = len(X)
    n_test = int(n * test_size)
    
    indices = list(range(n))
    
    if random_state is not None:
        a, c, m = 1103515245, 12345, 2**31
        seed = random_state
        for i in range(n - 1, 0, -1):
            seed = (a * seed + c) % m
            j = seed % (i + 1)
            indices[i], indices[j] = indices[j], indices[i]
    
    test_idx = set(indices[:n_test])
    
    X_train = [X[i] for i in range(n) if i not in test_idx]
    X_test = [X[i] for i in range(n) if i in test_idx]
    y_train = [y[i] for i in range(n) if i not in test_idx]
    y_test = [y[i] for i in range(n) if i in test_idx]
    
    return X_train, X_test, y_train, y_test


def accuracy_score(y_true, y_pred):
    """Calculate accuracy."""
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    return correct / len(y_true)


def precision_score(y_true, y_pred):
    """Calculate macro precision."""
    classes = list(set(y_true) | set(y_pred))
    precisions = []
    
    for cls in classes:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p == cls)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != cls and p == cls)
        precisions.append(tp / (tp + fp) if (tp + fp) > 0 else 0)
    
    return sum(precisions) / len(precisions)


def recall_score(y_true, y_pred):
    """Calculate macro recall."""
    classes = list(set(y_true) | set(y_pred))
    recalls = []
    
    for cls in classes:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p == cls)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p != cls)
        recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    
    return sum(recalls) / len(recalls)


def f1_score(y_true, y_pred):
    """Calculate macro F1 score."""
    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0


def confusion_matrix(y_true, y_pred):
    """Calculate confusion matrix."""
    classes = sorted(set(y_true) | set(y_pred))
    n = len(classes)
    idx = {c: i for i, c in enumerate(classes)}
    
    matrix = [[0] * n for _ in range(n)]
    for t, p in zip(y_true, y_pred):
        matrix[idx[t]][idx[p]] += 1
    
    return matrix, classes


def print_confusion_matrix(y_true, y_pred):
    """Print confusion matrix."""
    matrix, classes = confusion_matrix(y_true, y_pred)
    
    print(f"{'':>10}", end="")
    for c in classes:
        print(f"{c:>8}", end="")
    print()
    
    for i, c in enumerate(classes):
        print(f"{c:>10}", end="")
        for j in range(len(classes)):
            print(f"{matrix[i][j]:>8}", end="")
        print()


def classification_report(y_true, y_pred):
    """Generate classification report."""
    classes = sorted(set(y_true) | set(y_pred))
    
    lines = [f"{'Class':<12} {'Prec':<10} {'Recall':<10} {'F1':<10} {'Support':<10}"]
    lines.append("-" * 52)
    
    for cls in classes:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p == cls)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != cls and p == cls)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p != cls)
        support = sum(1 for t in y_true if t == cls)
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        
        lines.append(f"{cls:<12} {prec:<10.4f} {rec:<10.4f} {f1:<10.4f} {support:<10}")
    
    lines.append("-" * 52)
    lines.append(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    
    return "\n".join(lines)

In [3]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f"Размер обучающей выборки: {len(X_train)}")
print(f"Размер валидационной выборки: {len(X_val)}")
print(f"Размер тестовой выборки: {len(X_test)}")

Размер обучающей выборки: 535
Размер валидационной выборки: 178
Размер тестовой выборки: 178


## Пропущенные значения

Добавим пропущенные значения в данные для демонстрации работы алгоритма.

In [4]:
import math

X_train_missing = [row[:] for row in X_train]

missing_count = 0
for i in range(len(X_train_missing)):
    for j in range(len(X_train_missing[i])):
        if (i * 4 + j) % 10 == 0:
            X_train_missing[i][j] = None
            missing_count += 1

print(f"Добавлено пропущенных значений: {missing_count}")
print(f"Процент пропусков: {missing_count / (len(X_train_missing) * 4) * 100:.1f}%")

print("\nПример первых 5 строк с пропусками:")
for i, row in enumerate(X_train_missing[:5]):
    print(f"  {i}: {row}")

Добавлено пропущенных значений: 321
Процент пропусков: 15.0%

Пример первых 5 строк с пропусками:
  0: [None, 'male', 22.0, 1, 7.25, 'S']
  1: [3, 'female', 26.0, 0, 7.925, 'S']
  2: [1, 'female', None, 1, 53.1, 'S']
  3: [3, 'male', 35.0, 0, 8.05, 'S']
  4: [3, 'male', 2.0, 3, None, 'S']


In [17]:
clf_missing = DecisionTree(max_depth=10)
clf_missing.fit(X_train_missing, y_train, feature_types=feature_types)

y_pred_missing = clf_missing.predict(X_test)
acc_missing = accuracy_score(y_test, y_pred_missing)

print(f"\nТочность модели с пропусками: {acc_missing:.4f}")

x_demo = [3, 'male', None, 0, 7.25, None]
pred_demo = clf_missing.predict([x_demo])[0]
print(f"Пример объекта с пропусками: {x_demo}")
print(f"Предсказание: {class_names[pred_demo]}")


Точность модели с пропусками: 0.6011
Пример объекта с пропусками: [3, 'male', None, 0, 7.25, None]
Предсказание: Did not survive


## Обучение дерева

Обучим полное дерево на исходных данных.

In [18]:
clf_full = DecisionTree(max_depth=10)
clf_full.fit(X_train, y_train, feature_types=feature_types)

stats_full = clf_full.get_stats()

print(f"Количество узлов: {stats_full['nodes']}")
print(f"Количество листьев: {stats_full['leaves']}")
print(f"Максимальная глубина: {stats_full['depth']}")
print(f"\nТипы признаков: {sum(1 for t in feature_types.values() if t == 'categorical')} categorical, "
      f"{sum(1 for t in feature_types.values() if t == 'numeric')} numeric")

Количество узлов: 982
Количество листьев: 520
Максимальная глубина: 11

Типы признаков: 2 categorical, 4 numeric


## Оценка качества

In [19]:
y_pred_full = clf_full.predict(X_test)

print("Метрики качества")
print(f"Accuracy (точность): {accuracy_score(y_test, y_pred_full):.4f}")
print(f"Precision (полнота): {precision_score(y_test, y_pred_full):.4f}")
print(f"Recall (чувствительность): {recall_score(y_test, y_pred_full):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_full):.4f}")
print()
print(classification_report(y_test, y_pred_full))

Метрики качества
Accuracy (точность): 0.7416
Precision (полнота): 0.7477
Recall (чувствительность): 0.7169
F1-Score: 0.7320

Class        Prec       Recall     F1         Support   
----------------------------------------------------
0            0.7317     0.8738     0.7965     103       
1            0.7636     0.5600     0.6462     75        
----------------------------------------------------
Accuracy: 0.7416


In [23]:
print("Матрица ошибок (до редукции):")
print_confusion_matrix(y_test, y_pred_full)

Матрица ошибок (до редукции):
                 0       1
         0      90      13
         1      33      42


## Обрезка дерева

In [22]:
clf_pruned = clf_full.copy()

stats_before = clf_pruned.get_stats()
acc_train_before = clf_pruned.score(X_train, y_train)
acc_val_before = clf_pruned.score(X_val, y_val)
acc_test_before = clf_pruned.score(X_test, y_test)

print("До редукции:")
print(f"  Узлов: {stats_before['nodes']}, Листьев: {stats_before['leaves']}, Глубина: {stats_before['depth']}")
print(f"  Accuracy train: {acc_train_before:.4f}")
print(f"  Accuracy val:   {acc_val_before:.4f}")
print(f"  Accuracy test:  {acc_test_before:.4f}")

До редукции:
  Узлов: 982, Листьев: 520, Глубина: 11
  Accuracy train: 0.9065
  Accuracy val:   0.7865
  Accuracy test:  0.7416


In [20]:
clf_pruned.prune(X_val, y_val)

stats_after = clf_pruned.get_stats()
acc_train_after = clf_pruned.score(X_train, y_train)
acc_val_after = clf_pruned.score(X_val, y_val)
acc_test_after = clf_pruned.score(X_test, y_test)

print("\nПосле редукции:")
print(f"  Узлов: {stats_after['nodes']}, Листьев: {stats_after['leaves']}, Глубина: {stats_after['depth']}")
print(f"  Accuracy train: {acc_train_after:.4f}")
print(f"  Accuracy val:   {acc_val_after:.4f}")
print(f"  Accuracy test:  {acc_test_after:.4f}")


После редукции:
  Узлов: 52, Листьев: 28, Глубина: 10
  Accuracy train: 0.8355
  Accuracy val:   0.8427
  Accuracy test:  0.7247


## Сравнение до и после редукции

In [11]:
y_pred_pruned = clf_pruned.predict(X_test)


print(f"\n{'Метрика':<25} {'До редукции':<20} {'После редукции':<20}")
print("-" * 65)
print(f"{'Узлов':<25} {stats_before['nodes']:<20} {stats_after['nodes']:<20}")
print(f"{'Листьев':<25} {stats_before['leaves']:<20} {stats_after['leaves']:<20}")
print(f"{'Глубина':<25} {stats_before['depth']:<20} {stats_after['depth']:<20}")
print("-" * 65)
print(f"{'Accuracy (train)':<25} {acc_train_before:<20.4f} {acc_train_after:<20.4f}")
print(f"{'Accuracy (val)':<25} {acc_val_before:<20.4f} {acc_val_after:<20.4f}")
print(f"{'Accuracy (test)':<25} {acc_test_before:<20.4f} {acc_test_after:<20.4f}")
print("-" * 65)
print(f"{'Precision':<25} {precision_score(y_test, y_pred_full):<20.4f} {precision_score(y_test, y_pred_pruned):<20.4f}")
print(f"{'Recall':<25} {recall_score(y_test, y_pred_full):<20.4f} {recall_score(y_test, y_pred_pruned):<20.4f}")
print(f"{'F1-Score':<25} {f1_score(y_test, y_pred_full):<20.4f} {f1_score(y_test, y_pred_pruned):<20.4f}")


Метрика                   До редукции          После редукции      
-----------------------------------------------------------------
Узлов                     982                  52                  
Листьев                   520                  28                  
Глубина                   11                   10                  
-----------------------------------------------------------------
Accuracy (train)          0.9065               0.8355              
Accuracy (val)            0.7865               0.8427              
Accuracy (test)           0.7416               0.7247              
-----------------------------------------------------------------
Precision                 0.7477               0.7246              
Recall                    0.7169               0.7023              
F1-Score                  0.7320               0.7133              


In [24]:
print("\nМатрица ошибок (после редукции):")
print_confusion_matrix(y_test, y_pred_pruned)

reduction_nodes = (1 - stats_after['nodes'] / stats_before['nodes']) * 100
reduction_leaves = (1 - stats_after['leaves'] / stats_before['leaves']) * 100

print(f"\nСокращение количества узлов: {reduction_nodes:.1f}%")
print(f"Сокращение количества листьев: {reduction_leaves:.1f}%")


Матрица ошибок (после редукции):
                 0       1
         0      87      16
         1      33      42

Сокращение количества узлов: 94.7%
Сокращение количества листьев: 94.6%


In [13]:
from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTree
from sklearn.metrics import accuracy_score as sklearn_accuracy
from sklearn.metrics import precision_score as sklearn_precision
from sklearn.metrics import recall_score as sklearn_recall
from sklearn.metrics import f1_score as sklearn_f1
from sklearn.metrics import classification_report as sklearn_report
from sklearn.preprocessing import LabelEncoder

def encode_features(X, encoders=None):
    """Encode categorical features using LabelEncoder."""
    X_encoded = []
    if encoders is None:
        encoders = {}
    
    for row in X:
        new_row = []
        for i, val in enumerate(row):
            if feature_types.get(i) == 'categorical':
                if i not in encoders:
                    encoders[i] = LabelEncoder()
                    all_vals = list(set(r[i] for r in X if r[i] is not None))
                    encoders[i].fit(all_vals)
                new_row.append(encoders[i].transform([val])[0] if val is not None else 0)
            else:
                new_row.append(val if val is not None else 0)
        X_encoded.append(new_row)
    return X_encoded, encoders

X_train_enc, encoders = encode_features(X_train)
X_test_enc, _ = encode_features(X_test, encoders)

sklearn_clf = SklearnDecisionTree(
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

sklearn_clf.fit(X_train_enc, y_train)
y_pred_sklearn = sklearn_clf.predict(X_test_enc)

print(f"Количество листьев: {sklearn_clf.get_n_leaves()}")
print(f"Глубина дерева: {sklearn_clf.get_depth()}")

Количество листьев: 121
Глубина дерева: 20


## Сравнение со sklearn

In [21]:

print(f"{'Листьев':<25} {stats_before['leaves']:<18} {stats_after['leaves']:<18} {sklearn_clf.get_n_leaves():<18}")
print(f"{'Глубина':<25} {stats_before['depth']:<18} {stats_after['depth']:<18} {sklearn_clf.get_depth():<18}")

acc_our_full = accuracy_score(y_test, y_pred_full)
acc_our_pruned = accuracy_score(y_test, y_pred_pruned)
acc_sklearn = sklearn_accuracy(y_test, y_pred_sklearn)
print(f"{'Accuracy':<25} {acc_our_full:<18.4f} {acc_our_pruned:<18.4f} {acc_sklearn:<18.4f}")

prec_our_full = precision_score(y_test, y_pred_full)
prec_our_pruned = precision_score(y_test, y_pred_pruned)
prec_sklearn = sklearn_precision(y_test, y_pred_sklearn, average='macro')
print(f"{'Precision (macro)':<25} {prec_our_full:<18.4f} {prec_our_pruned:<18.4f} {prec_sklearn:<18.4f}")

rec_our_full = recall_score(y_test, y_pred_full)
rec_our_pruned = recall_score(y_test, y_pred_pruned)
rec_sklearn = sklearn_recall(y_test, y_pred_sklearn, average='macro')
print(f"{'Recall (macro)':<25} {rec_our_full:<18.4f} {rec_our_pruned:<18.4f} {rec_sklearn:<18.4f}")

f1_our_full = f1_score(y_test, y_pred_full)
f1_our_pruned = f1_score(y_test, y_pred_pruned)
f1_sklearn = sklearn_f1(y_test, y_pred_sklearn, average='macro')
print(f"{'F1-Score (macro)':<25} {f1_our_full:<18.4f} {f1_our_pruned:<18.4f} {f1_sklearn:<18.4f}")

Листьев                   520                28                 121               
Глубина                   11                 10                 20                
Accuracy                  0.7416             0.7247             0.7247            
Precision (macro)         0.7477             0.7246             0.7179            
Recall (macro)            0.7169             0.7023             0.7186            
F1-Score (macro)          0.7320             0.7133             0.7182            


In [25]:
print(sklearn_report(y_test, y_pred_sklearn, target_names=list(class_names.values())))

                 precision    recall  f1-score   support

Did not survive       0.76      0.76      0.76       103
       Survived       0.67      0.68      0.68        75

       accuracy                           0.72       178
      macro avg       0.72      0.72      0.72       178
   weighted avg       0.73      0.72      0.72       178

